# Track 1 reconstructed episode 1
Restore a hash-verified runtime and reconstruct the full original timeline.

In [ ]:
import os,sys,json,subprocess,urllib.request,hashlib,tarfile,shutil
from pathlib import Path
def file_sha256(path):
 digest=hashlib.sha256()
 with path.open('rb') as handle:
  for chunk in iter(lambda:handle.read(16*1024*1024),b''):digest.update(chunk)
 return digest.hexdigest()
manifests=list(Path('/kaggle/input').rglob('runtime_cache_manifest.json'))
assert len(manifests)==1,manifests
manifest=json.loads(manifests[0].read_text())
cache=manifests[0].with_name('track1_runtime.bin')
if not cache.exists():cache=manifests[0].with_name('track1_runtime.tar.gz')
assert cache.stat().st_size==manifest['bytes']
assert file_sha256(cache)==manifest['sha256']
assert manifest['runtime_revision']=='8c778f905cb35f76f8e6cd6e2ea486da4fb9e0ef'
subprocess.run(['apt-get','update','-qq'],check=True)
subprocess.run(['apt-get','install','-y','-qq','gcc-12','g++-12','cmake','libeigen3-dev','libboost-system-dev','libboost-program-options-dev','libegl1-mesa-dev','libosmesa6-dev','libspatialindex-dev','libgl1','ffmpeg'],check=True)
# Only accept the explicit paths listed in the manifest, preventing arbitrary writes.
allowed=tuple(x.lstrip('/').rstrip('/')+'/' for x in manifest['paths'])
with tarfile.open(cache) as archive:
 for item in archive.getmembers():
  name=item.name.rstrip('/')+'/'
  if not name.startswith(allowed):raise ValueError('Unexpected runtime archive member '+item.name)
 archive.extractall('/',filter='fully_trusted')
py='/tmp/track1-env/bin/python';cuda=Path('/tmp/cuda124')
modules=Path('/tmp/video_to_data/reconstruction/modules')
workspace=Path('/workspace');workspace.mkdir(exist_ok=True)
for name in ['v2d_common','v2d_mv','v2d_mesh','v2d_foundation_pose','v2d_moge','v2d_sam3d_body','v2d_cari4d']:
 path=workspace/name
 if not path.exists():path.symlink_to(modules/name,target_is_directory=True)
env=os.environ.copy();env.update(CUDA_HOME=str(cuda),PATH=str(cuda/'bin')+':'+env['PATH'],CC='gcc-12',CXX='g++-12',TORCH_CUDA_ARCH_LIST='7.5',FORCE_CUDA='1',MAX_JOBS='2',PYOPENGL_PLATFORM='egl',OPENCV_IO_ENABLE_OPENEXR='1',CUB_HOME=str(cuda/'include'),CPATH=str(cuda/'include')+':/usr/include/eigen3')
env['PYTHONPATH']=':'.join(str(workspace/x) for x in ['v2d_cari4d/lib/cari4d','v2d_sam3d_body/lib','v2d_foundation_pose/lib/FoundationPose'])
print('Verified native runtime cache restored',flush=True)


In [ ]:
test="""
import json, torch, kaolin, pytorch3d, nvdiffrast.torch as dr
from pytorch3d.ops import knn_points
from pathlib import Path
from v2d.cari4d.lib.run_inference import run_inference
x=torch.rand(1,32,3,device='cuda');r=knn_points(x,x,K=1)
assert r.dists.max().item()<1e-6
ctx=dr.RasterizeCudaContext()
report={'torch':torch.__version__,'cuda':torch.version.cuda,'gpu':torch.cuda.get_device_name(0),'knn_gpu':True,'rasterizer_context':True,'cari4d_import':True}
Path('/kaggle/working/track1_runtime_report.json').write_text(json.dumps(report,indent=2));print(report)
"""
subprocess.run([py,'-c',test],env=env,check=True)


In [ ]:
weights=Path('/tmp/track1_weights');weights.mkdir(exist_ok=True)
manifests=[json.loads(p.read_text()) for p in Path('/kaggle/input').rglob('model_manifest.json')]
manifest=next(value for value in manifests if isinstance(value,list) and len(value)==8 and all('file' in x and 'sha256' in x for x in value))
archives=list(Path('/kaggle/input').rglob('track1_models.tar'))
if len(archives)==1:
 with tarfile.open(archives[0]) as archive:archive.extractall(weights,filter='data')
else:
 # Kaggle expands uploaded .tar files into a directory named after the archive.
 roots=[p.parents[2] for p in Path('/kaggle/input').rglob('step200000.pth') if p.parent.name=='2026-08-25-09-35-57']
 assert len(roots)==1,roots
 for item in manifest:
  source=roots[0]/item['file'];destination=weights/item['file']
  destination.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(source,destination)
for item in manifest:
 path=weights/item['file'];assert path.stat().st_size==item['bytes']
 assert file_sha256(path)==item['sha256'],path
# T4 lacks bfloat16. Use full precision for the human initializer to avoid FP16 overflow.
config=weights/'sam3d_body/checkpoints/sam-3d-body-dinov3/model_config.yaml'
original_config=config.read_text()
assert 'USE_FP16: true' in original_config
config.write_text(original_config.replace('USE_FP16: true','USE_FP16: false'))
Path('/kaggle/working/t4_precision_manifest.json').write_text(json.dumps({
 'sam3d_precision':'float32','original_config_sha256':hashlib.sha256(original_config.encode()).hexdigest(),
 'effective_config_sha256':hashlib.sha256(config.read_bytes()).hexdigest()},indent=2))
env['HF_HOME']=str(weights/'hf_home')
public_models="""
from pathlib import Path
from v2d.cari4d.lib.download_weights import _download_moge2,_download_torch_hub_models
from v2d.foundation_pose.lib.download_weights import download_weights
from v2d.foundation_pose.lib.backends import NVLABS_PYTORCH
weights=Path('/tmp/track1_weights')
_download_moge2(weights)
_download_torch_hub_models(weights)
download_weights(str(weights/'foundationpose'),backend=NVLABS_PYTORCH)
"""
public_caches=list(Path('/kaggle/input').rglob('public_models.bin'))
if public_caches:
 assert len(public_caches)==1,public_caches
 archive_path=public_caches[0]
 cache_manifest=json.loads(archive_path.with_name('public_model_cache_manifest.json').read_text())
 assert archive_path.stat().st_size==cache_manifest['bytes']
 assert file_sha256(archive_path)==cache_manifest['sha256']
 prefixes=tuple(x.rstrip('/')+'/' for x in cache_manifest['paths'])
 with tarfile.open(archive_path) as archive:
  for item in archive.getmembers():
   if not (item.name.rstrip('/')+'/').startswith(prefixes):raise ValueError('Unexpected public-model cache member')
  archive.extractall(weights,filter='data')
 print('Verified public model cache restored',flush=True)
else:
 subprocess.run([py,'-c',public_models],env=env,check=True)


In [ ]:
code=Path('/tmp/track1_code');code.mkdir(exist_ok=True)
(code/'fit_hoop.py').write_text('"""Fit a circular tube shape prior to an automatically segmented hoop image.\n\nThe released object class supplies circular topology; the observed mask estimates\nits projected axes and tube-to-ring radius. Scale and 3D pose remain unestimated.\n"""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport numpy as np\nfrom PIL import Image\nfrom scipy import ndimage\nfrom scipy.optimize import least_squares\nimport trimesh\n\n\ndef main():\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--image\',type=Path,required=True)\n    p.add_argument(\'--output\',type=Path,required=True)\n    a=p.parse_args()\n    image=np.asarray(Image.open(a.image).convert(\'RGBA\'))\n    mask=image[:,:,3]>127\n    y,x=np.where(mask)\n    if len(x)<50 or mask.mean()>.6:\n        raise ValueError(\'Expected a segmented thin hoop, not a filled object/background\')\n    xy=np.column_stack([x,y]).astype(float)\n    center=np.median(xy,axis=0)\n    covariance=np.cov(xy.T)\n    values,vectors=np.linalg.eigh(covariance)\n    axes=np.sqrt(2*values[::-1]);direction=vectors[:,::-1]\n    angle=np.arctan2(direction[1,0],direction[0,0])\n\n    def residual(parameters):\n        cx,cy,log_a,log_b,theta=parameters\n        c,s=np.cos(theta),np.sin(theta)\n        q=(xy-[cx,cy])@np.array([[c,-s],[s,c]])\n        return np.sqrt((q[:,0]/np.exp(log_a))**2+(q[:,1]/np.exp(log_b))**2)-1\n\n    fit=least_squares(residual,[*center,*np.log(axes),angle],loss=\'soft_l1\',f_scale=.03)\n    if not fit.success:raise ValueError(\'Projected ellipse fit failed\')\n    major=float(max(np.exp(fit.x[2:4])))\n    distance=ndimage.distance_transform_edt(mask)\n    ridge=mask&(distance==ndimage.maximum_filter(distance,size=3))\n    # Pixel-centre distances overestimate tube radius by half a pixel.\n    tube_pixels=max(.5,float(np.median(distance[ridge]))-.5)\n    ratio=tube_pixels/major\n    if not .002<ratio<.15:raise ValueError(f\'Implausible observed hoop tube ratio {ratio}\')\n    mesh=trimesh.creation.torus(major_radius=1.,minor_radius=ratio,\n                               major_sections=128,minor_sections=16)\n    colour=np.median(image[:,:,:3][mask],axis=0).astype(np.uint8)\n    mesh.visual.vertex_colors=np.tile(np.r_[colour,255],(len(mesh.vertices),1))\n    a.output.mkdir(parents=True,exist_ok=True)\n    mesh.export(a.output/\'normalized_shape.glb\')\n    report={\'method\':\'mask-fitted circular tube prior\',\'class\':\'hula_hoop\',\n        \'image_sha256\':hashlib.sha256(a.image.read_bytes()).hexdigest(),\n        \'mask_pixels\':int(mask.sum()),\'alpha_coverage\':float(mask.mean()),\n        \'ellipse_parameters\':fit.x.tolist(),\'ellipse_residual_median\':float(np.median(np.abs(fit.fun))),\n        \'major_radius_pixels\':major,\'tube_radius_pixels\':tube_pixels,\n        \'tube_to_major_radius\':ratio,\'vertices\':len(mesh.vertices),\'faces\':len(mesh.faces),\n        \'metric_scale_estimated\':False,\'camera_registration_estimated\':False,\n        \'manual_geometry_used\':False,\'ground_truth_used\':False,\'ready_for_submission\':False}\n    (a.output/\'shape_report.json\').write_text(json.dumps(report,indent=2)+\'\\n\')\n    print(json.dumps(report,indent=2))\n\n\nif __name__==\'__main__\':main()\n')
(code/'calibrate_object.py').write_text('"""Estimate metric mesh scale/pose from Track 1 RGB and human-aligned depth.\n\nInputs are an actual image-conditioned mesh and an observed-frame NPZ containing\nrgb, depth, human_mask, object_mask and K. FoundationPose registration proposes\npose at each scale; observed surface agreement and occlusion-aware silhouette\nselect the hypothesis. No clean trajectory or external object template is used.\n"""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport numpy as np\nfrom scipy.spatial import cKDTree\nimport trimesh\nfrom prep.run_foundationpose_mhr_export import (\n    _pose_estimator, _prepare_foundationpose_inputs, _render_candidate_masks)\n\n\ndef main():\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--mesh\',type=Path,required=True)\n    p.add_argument(\'--frame-data\',type=Path,required=True)\n    p.add_argument(\'--output\',type=Path,required=True)\n    a=p.parse_args()\n    a.output.mkdir(parents=True,exist_ok=True)\n    with np.load(a.frame_data,allow_pickle=False) as f:\n        rgb,depth,human,mask,K=_prepare_foundationpose_inputs(\n            f[\'rgb\'],f[\'depth\'],f[\'human_mask\'],f[\'object_mask\'],f[\'K\'])\n    visible=mask.astype(bool)&~human.astype(bool)&np.isfinite(depth)&(depth>0)\n    ys,xs=np.where(visible)\n    if len(xs)<50:raise ValueError(\'Insufficient observed, unoccluded object depth\')\n    depth_values=depth[ys,xs]\n    cloud=np.column_stack([(xs-K[0,2])*depth_values/K[0,0],\n                           (ys-K[1,2])*depth_values/K[1,1],depth_values])\n    extent=np.percentile(cloud,95,axis=0)-np.percentile(cloud,5,axis=0)\n    mesh=trimesh.load(a.mesh,force=\'mesh\')\n    if not isinstance(mesh,trimesh.Trimesh) or not len(mesh.faces):\n        raise ValueError(\'Expected a reconstructed triangle mesh\')\n    initial=float(np.linalg.norm(extent)/max(mesh.extents))\n    if not np.isfinite(initial) or initial<=0:raise ValueError(\'Invalid scale initialization\')\n    rng=np.random.default_rng(42)\n    cloud=cloud[rng.choice(len(cloud),min(2048,len(cloud)),replace=False)]\n    candidates=[];estimator=None\n\n    def evaluate(scale):\n        nonlocal estimator\n        candidate=mesh.copy();candidate.vertices*=scale\n        if estimator is None:\n            estimator=_pose_estimator(candidate,a.output/\'registration_debug\')\n        else:\n            estimator.reset_object(model_pts=np.asarray(candidate.vertices),\n                model_normals=np.asarray(candidate.vertex_normals),mesh=candidate)\n        pose=np.asarray(estimator.register(K=K,rgb=rgb,depth=depth,\n            ob_mask=visible,iteration=5),dtype=np.float32)\n        if pose.shape!=(4,4) or not np.isfinite(pose).all():\n            raise ValueError(\'Invalid registered pose\')\n        obj_points=(cloud-pose[:3,3])@pose[:3,:3]\n        distances,_=cKDTree(np.asarray(candidate.vertices)).query(obj_points)\n        cutoff=np.quantile(distances,.8)\n        surface_error=float(distances[distances<=cutoff].mean())\n        centered=pose@np.linalg.inv(estimator.get_tf_to_centered_mesh().detach().cpu().numpy())\n        rendered=_render_candidate_masks(estimator,centered[None],K,depth.shape,chunk_size=1)[0]\n        rendered&=~human.astype(bool)\n        union=np.logical_or(rendered,visible).sum()\n        iou=float(np.logical_and(rendered,visible).sum()/max(1,union))\n        score=surface_error+.03*(1-iou)\n        result={\'scale\':float(scale),\'pose\':pose.tolist(),\'surface_error_m\':surface_error,\n                \'visible_mask_iou\':iou,\'selection_cost\':score}\n        candidates.append(result)\n        print(json.dumps(result),flush=True)\n        return result\n\n    for factor in np.geomspace(.6,1.6,7):evaluate(initial*factor)\n    coarse=min(candidates,key=lambda x:x[\'selection_cost\'])\n    for factor in np.linspace(.85,1.15,5):evaluate(coarse[\'scale\']*factor)\n    best=min(candidates,key=lambda x:x[\'selection_cost\'])\n    mesh.vertices*=best[\'scale\']\n    # Project observed appearance onto visible vertices; unseen vertices use the\n    # observed object\'s median colour rather than background pixels.\n    pose=np.asarray(best[\'pose\']);camera=np.asarray(mesh.vertices)@pose[:3,:3].T+pose[:3,3]\n    positive=camera[:,2]>1e-6\n    u=np.zeros(len(camera),dtype=int);v=u.copy()\n    u[positive]=np.rint(K[0,0]*camera[positive,0]/camera[positive,2]+K[0,2]).astype(int)\n    v[positive]=np.rint(K[1,1]*camera[positive,1]/camera[positive,2]+K[1,2]).astype(int)\n    inside=positive&(u>=0)&(u<rgb.shape[1])&(v>=0)&(v<rgb.shape[0])\n    colour=np.median(rgb[visible],axis=0).astype(np.uint8)\n    colours=np.tile(np.r_[colour,255],(len(camera),1)).astype(np.uint8)\n    ids=np.flatnonzero(inside)\n    visible_vertices=ids[visible[v[ids],u[ids]]&\n        (np.abs(depth[v[ids],u[ids]]-camera[ids,2])<.03)]\n    colours[visible_vertices,:3]=rgb[v[visible_vertices],u[visible_vertices]]\n    mesh.visual=trimesh.visual.ColorVisuals(vertex_colors=colours)\n    mesh.export(a.output/\'metric_mesh.glb\')\n    np.save(a.output/\'initial_camera_pose.npy\',pose)\n    report={\'initial_scale\':initial,\'selected\':best,\'candidates\':candidates,\n        \'selection\':\'observed trimmed surface distance + 0.03m*(1-visible-mask-IoU)\',\n        \'manual_geometry_used\':False,\'ground_truth_used\':False,\n        \'frame_data_sha256\':hashlib.sha256(a.frame_data.read_bytes()).hexdigest()}\n    (a.output/\'scale_report.json\').write_text(json.dumps(report,indent=2)+\'\\n\')\n\n\nif __name__==\'__main__\':main()\n')
(code/'run_episode.py').write_text('"""Run actual Track 1 reconstruction with observed-depth object calibration.\n\nA bootstrap pass estimates human geometry and metric depth. A separate final\nexport uses the calibrated object; unchanged camera/human/depth artifacts are\ntransferred explicitly, with their source identities retained in the run report.\n"""\nimport argparse\nimport json\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport sys\nimport numpy as np\nfrom v2d.cari4d.lib import run_inference as native\n\n\ndef main():\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--video\',type=Path,required=True)\n    p.add_argument(\'--masks\',type=Path,required=True)\n    p.add_argument(\'--mesh\',type=Path,required=True)\n    p.add_argument(\'--weights\',type=Path,required=True)\n    p.add_argument(\'--output\',type=Path,required=True)\n    p.add_argument(\'--episode\',type=int,required=True)\n    p.add_argument(\'--frames\',type=int,required=True)\n    p.add_argument(\'--seed-frame\',type=int,default=15)\n    a=p.parse_args()\n    original=native._stage_run\n    bootstrap_outputs={}\n\n    class BootstrapComplete(Exception):pass\n\n    def bootstrap(name,command,inputs,outputs,marker_root,env,overwrite):\n        if name==\'05_object_pose\':raise BootstrapComplete()\n        result=original(name,command,inputs,outputs,marker_root,env,overwrite)\n        bootstrap_outputs[name]=list(outputs)\n        return result\n\n    options=dict(download_models=False,expected_frames=a.frames,moge_batch_size=2,\n        sam3d_chunk_size=1,sam3d_refit_batch_size=32,\n        depth_alignment_render_batch_size=1,depth_alignment_encoding_workers=2,\n        depth_alignment_write_batch_size=16,coconet_render_batch_size=1,\n        coconet_crop_workers=2,coconet_crop_buffer_count=1,\n        postopt_batch_size=32,render_batch_size=1)\n    native._stage_run=bootstrap\n    try:\n        native.run_inference(str(a.video),str(a.masks),str(a.mesh),str(a.weights),\n                             str(a.output/\'bootstrap\'),**options)\n    except BootstrapComplete:\n        pass\n    finally:\n        native._stage_run=original\n    sequence=a.video.name[:-len(\'.0.color.mp4\')]\n    source_root=a.output/\'bootstrap\'/sequence\n    export=source_root/\'export\'/sequence\n    from prep.mhr_export_utils import (camera_calibration,frame_names,load_edex,\n                                      read_depth_m,read_rgb,read_mask)\n    K,_=camera_calibration(load_edex(export),0)\n    names=frame_names(export,0)\n    if len(names)!=a.frames:raise ValueError(\'Bootstrap changed the video timeline\')\n    frame=names[min(a.seed_frame,a.frames-1)]\n    calibration=a.output/\'calibration\';calibration.mkdir(parents=True,exist_ok=True)\n    frame_data=calibration/\'observed_frame.npz\'\n    np.savez_compressed(frame_data,rgb=read_rgb(export,0,frame),\n        depth=read_depth_m(export,0,frame,depth_root=source_root/\'depth/moge2/aligned.h5\'),\n        human_mask=read_mask(export,\'human\',0,frame),\n        object_mask=read_mask(export,\'object\',0,frame),K=K)\n    subprocess.run([sys.executable,str(Path(__file__).with_name(\'calibrate_object.py\')),\n        \'--mesh\',str(a.mesh),\'--frame-data\',str(frame_data),\'--output\',str(calibration)],check=True)\n    transferred=[]\n\n    def final_stage(name,command,inputs,outputs,marker_root,env,overwrite):\n        if name not in {\'01_depth\',\'03_human_initialization\',\'04_depth_alignment\'}:\n            return original(name,command,inputs,outputs,marker_root,env,overwrite)\n        source=bootstrap_outputs[name]\n        if len(source)!=len(outputs):raise ValueError(\'Native stage output contract changed\')\n        for src,dst in zip(source,outputs):\n            dst.parent.mkdir(parents=True,exist_ok=True)\n            shutil.copy2(src,dst)\n        entry={\'name\':name,\'reused\':True,\'elapsed_seconds\':0.,\n            \'reuse_reason\':\'Identical video, masks and camera; object-independent stage\',\n            \'source_outputs\':native._output_identities(source),\n            \'outputs\':native._output_identities(outputs)}\n        transferred.append(entry)\n        return entry\n\n    native._stage_run=final_stage\n    try:\n        native.run_inference(str(a.video),str(a.masks),str(calibration/\'metric_mesh.glb\'),\n            str(a.weights),str(a.output/\'final\'),**options)\n    finally:\n        native._stage_run=original\n    (a.output/\'stage_transfer_report.json\').write_text(json.dumps(transferred,indent=2)+\'\\n\')\n    final=a.output/\'final\'/sequence\n    subprocess.run([sys.executable,str(Path(__file__).with_name(\'export_cari4d.py\')),\n        \'--bundle\',str(final/\'inference/refined.pth\'),\n        \'--mesh\',str(final/\'export\'/sequence/\'object_mesh/output_aligned.glb\'),\n        \'--mhr-assets-root\',str(a.weights/\'sam3d_body\'),\n        \'--episode\',str(a.episode),\'--frames\',str(a.frames),\n        \'--output\',str(a.output/\'episodes\')],check=True)\n\n\nif __name__==\'__main__\':main()\n')
(code/'export_cari4d.py').write_text('"""Export an actual native-MHR CARI4D wild-video result to the official kit.\n\nRun in the reconstruction environment with its PYTHONPATH and MHR assets.\nThis is an adapter, not a reconstruction method. It refuses incomplete timelines\nand validates the exported native parameters by decoding the reference model.\n"""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport shutil\nimport numpy as np\nimport torch\nfrom lib_mhr import MHRLayer\n\n\ndef sha256(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        while chunk := f.read(8*1024*1024):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef main():\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--bundle\', type=Path, required=True)\n    p.add_argument(\'--mesh\', type=Path, required=True)\n    p.add_argument(\'--mhr-assets-root\', type=Path, required=True)\n    p.add_argument(\'--episode\', type=int, required=True)\n    p.add_argument(\'--frames\', type=int, required=True)\n    p.add_argument(\'--output\', type=Path, required=True)\n    p.add_argument(\'--batch-size\', type=int, default=8)\n    p.add_argument(\'--device\', default=\'cuda\')\n    a = p.parse_args()\n    if a.episode < 0 or a.frames < 1 or a.batch_size < 1:\n        p.error(\'Episode, frame count or batch size is invalid\')\n    if a.mesh.name != \'output_aligned.glb\':\n        p.error(\'Use the actual wild-pipeline aligned mesh: output_aligned.glb\')\n    # This bundle is produced by our pinned inference code, not an untrusted upload.\n    bundle = torch.load(a.bundle, map_location=\'cpu\', weights_only=False)\n    if bundle.get(\'metadata\', {}).get(\'ground_truth_used\') is not False:\n        raise ValueError(\'Expected explicit no-ground-truth inference provenance\')\n    frames = [int(f) for f in bundle[\'frames\']]\n    if frames != list(range(a.frames)):\n        raise ValueError(\'Bundle must cover every original video frame in order\')\n    pr = bundle[\'pr\']\n    for key in (\'mhr_global_rot6d\', \'mhr_trans\', \'mhr_body_pose_cont\',\n                \'mhr_hand\', \'mhr_shape\', \'mhr_scale\'):\n        values = np.asarray(pr[key])\n        if len(values) != a.frames or not np.isfinite(values).all():\n            raise ValueError(f\'Invalid or incomplete parameter array: {key}\')\n    poses = np.asarray(pr[\'pose_abs\'], dtype=np.float32)\n    if poses.shape != (a.frames,4,4) or not np.isfinite(poses).all():\n        raise ValueError(\'Expected finite object poses [T,4,4]\')\n    np.testing.assert_allclose(poses[:,3,:], np.tile([0,0,0,1],(a.frames,1)), atol=1e-6)\n    rotations = poses[:,:3,:3]\n    np.testing.assert_allclose(rotations.transpose(0,2,1)@rotations,\n                               np.tile(np.eye(3),(a.frames,1,1)), atol=1e-4)\n    if (np.linalg.det(rotations) <= 0).any():\n        raise ValueError(\'Object pose contains a reflection\')\n    metadata = bundle.get(\'metadata\', {})\n    transform = np.asarray(metadata.get(\'object_mesh_to_training_transform\', np.eye(4)))\n    if not np.allclose(transform, np.eye(4), atol=1e-6):\n        raise ValueError(\'Nonidentity object mesh frame requires explicit geometric conversion\')\n\n    layer = MHRLayer.from_mhr_assets(mhr_assets_root=a.mhr_assets_root, device=a.device)\n    backend = layer.backend\n    model_path = backend.mhr_model_path\n    reference = torch.jit.load(str(model_path), map_location=a.device).eval()\n    flip = torch.tensor([1.,-1.,-1.], device=a.device)\n    # The competition requires one identity for the entire clip.\n    shape = np.median(np.asarray(pr[\'mhr_shape\']), axis=0).astype(np.float32)\n    latent_scale = np.median(np.asarray(pr[\'mhr_scale\']), axis=0).astype(np.float32)\n    exported = []\n    scales = None\n    errors = []\n    for start in range(0,a.frames,a.batch_size):\n        stop = min(start+a.batch_size,a.frames)\n        params = {k: torch.as_tensor(np.asarray(v)[start:stop],device=a.device,dtype=torch.float32)\n                  for k,v in pr.items() if k.startswith(\'mhr_\')}\n        count = stop-start\n        params[\'mhr_shape\'] = torch.as_tensor(shape,device=a.device).expand(count,-1)\n        params[\'mhr_scale\'] = torch.as_tensor(latent_scale,device=a.device).expand(count,-1)\n        # Expression is absent from the challenge schema and the scorer uses zero.\n        params[\'mhr_face\'] = torch.zeros(count,72,device=a.device)\n        with torch.inference_mode():\n            context = backend.prepare_vertices_context(params)\n            translation,body,identity,scale = backend._mutable_vertices_inputs(params,context)\n            zero = torch.zeros_like(translation)\n            verts, native = context.head.mhr_forward(\n                global_trans=zero,global_rot=context.global_rot,body_pose_params=body,\n                hand_pose_params=context.hand,scale_params=scale,shape_params=identity,\n                expr_params=params[\'mhr_face\'],return_model_params=True)\n            target = verts*flip + translation[:,None,:]\n            native = native.clone()\n            expr = torch.zeros(count,72,device=a.device)\n\n            def decode(model_params):\n                v,_ = reference(identity.contiguous(),model_params.contiguous(),expr,True)\n                return v*flip/100.\n\n            base = decode(native)\n            # Determine the native root translation units from the model itself.\n            columns = []\n            for axis in range(3):\n                shifted = native.clone();shifted[:,axis] += 1.\n                columns.append((decode(shifted)-base).mean(dim=1))\n            jacobian = torch.stack(columns,dim=-1)\n            delta = torch.linalg.solve(jacobian,(target-base).mean(dim=1).unsqueeze(-1)).squeeze(-1)\n            native[:,:3] += delta\n            actual = decode(native)\n            error = torch.linalg.vector_norm(actual-target,dim=-1).amax(dim=1)\n            if not torch.isfinite(error).all() or error.max().item() > 0.0005:\n                raise ValueError(f\'Native decoder round trip exceeds 0.5 mm: {error.max().item()} m\')\n        block = native.detach().cpu().numpy()\n        if block.shape != (count,204):\n            raise ValueError(f\'Expected native MHR parameters [B,204], got {block.shape}\')\n        if scales is None:\n            scales = block[0,136:].copy()\n        np.testing.assert_allclose(block[:,136:],np.tile(scales,(count,1)),atol=1e-6)\n        exported.append(block[:,:136]);errors.extend(error.cpu().tolist())\n    a.output.mkdir(parents=True,exist_ok=True)\n    stem = a.output/f\'episode_{a.episode:06d}\'\n    np.savez_compressed(stem.with_suffix(\'.npz\'),pose=np.concatenate(exported),\n        scales=scales,shape=shape,object_rotation=rotations,\n        object_translation=poses[:,:3,3],object_scale=np.float32(1.))\n    mesh_out = a.output/(stem.name+\'_object.glb\')\n    shutil.copy2(a.mesh,mesh_out)\n    report = {\'episode\':a.episode,\'frames\':a.frames,\'source_bundle_sha256\':sha256(a.bundle),\n        \'mesh_sha256\':sha256(mesh_out),\'mhr_model_sha256\':sha256(model_path),\n        \'identity_selection\':\'per-clip median of predicted identity latents\',\n        \'expression\':\'zero, matching challenge scorer\',\n        \'max_native_round_trip_error_m\':max(errors),\'ground_truth_used\':False}\n    stem.with_suffix(\'.report.json\').write_text(json.dumps(report,indent=2)+\'\\n\')\n    print(json.dumps(report,indent=2))\n\n\nif __name__ == \'__main__\':\n    main()\n')
inputs=Path('/kaggle/input')
episode_id=1
folder=f'episode_{episode_id:06d}'
image=next(p for p in inputs.rglob('object_observation.png') if p.parent.name==folder)
masks=next(p for p in inputs.rglob('masks.h5') if p.parent.name==folder)
subprocess.run([py,str(code/'fit_hoop.py'),'--image',str(image),'--output','/kaggle/working/shape'],env=env,check=True)
base='https://huggingface.co/datasets/nvidia/video_to_data_challenge/resolve/main/track_1/'
with urllib.request.urlopen(base+'meta/episodes_metadata.jsonl') as response: metadata=[json.loads(line) for line in response.read().decode().splitlines()]
with urllib.request.urlopen(base+'meta/episodes.jsonl') as response: episodes=[json.loads(line) for line in response.read().decode().splitlines()]
m=next(x for x in metadata if x['episode_index']==episode_id)
episode=next(x for x in episodes if x['episode_index']==episode_id)
video=Path('/tmp')/(m['sequence_id']+'.0.color.mp4')
urllib.request.urlretrieve(base+f'videos/chunk-000/observation.images.exo_camera/episode_{episode_id:06d}.mp4',video)
env.update(TORCH_HOME=str(weights/'sam3d_body/torch_home'),MHR_ASSETS_ROOT=str(weights/'sam3d_body'),FOUNDATIONPOSE_WEIGHTS_DIR=str(weights/'foundationpose/nvlabs_pytorch'),HDF5_USE_FILE_LOCKING='FALSE',PYTHONUNBUFFERED='1')
subprocess.run([py,str(code/'run_episode.py'),'--video',str(video),'--masks',str(masks),
 '--mesh','/kaggle/working/shape/normalized_shape.glb','--weights',str(weights),
 '--seed-frame',str(json.loads(image.with_name('segmentation_report.json').read_text())['seed']['frame']),
 '--output','/kaggle/working/reconstruction','--episode',str(episode_id),'--frames',str(episode['length'])],env=env,check=True)
